# Municipality Modeling

Linking per-municipal reservoir counts/density to municipal attributes

In [ ]:
import pandas as pd
import geopandas as gpd
from sklearn import metrics
from sklearn import svm
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, train_test_split
import os
import numpy as np
import matplotlib.pyplot as plt
import pygam

In [ ]:
def read_process_csv_to_gdf(csv):
    temp_df = pd.read_csv(csv)
    temp_df['satellite'] = os.path.basename(csv)[:8]
    temp_df['year'] = int(os.path.basename(csv)[9:13])
    temp_df = temp_df.loc[temp_df['hydropoly_max']<=50]
    temp_df['area_ha'] = temp_df['area']*100/10000 # HA
    temp_df['area_km'] = temp_df['area']*100/(1000*1000) # km2
    temp_df = temp_df.loc[temp_df['area_ha']<=50] # Remove greater than 50 ha
    temp_df = temp_df.loc[temp_df['area_ha']>=0.05] # Remove less than 0.05 ha
    temp_gdf = gpd.GeoDataFrame(
        temp_df, geometry=gpd.points_from_xy(temp_df.longitude, temp_df.latitude),
        crs='EPSG:4326'
    )
    return temp_gdf

def sjoin_summarize(points_gdf, poly_gdf, poly_field):
    joined_gdf = gpd.sjoin(points_gdf, poly_gdf, predicate='within', how='inner')
    return joined_gdf[['res_area_ha', poly_field]].groupby(poly_field).agg(['sum', 'count', 'median'])['res_area_ha']

def sjoin_summarize_nogroup(points_gdf, poly_gdf):
    joined_gdf = gpd.sjoin(points_gdf, poly_gdf, predicate='within', how='inner')
    return joined_gdf[['area_ha']].agg(['sum', 'count', 'median'])

In [ ]:
in_csv = '../clean_summarize/out/sentinel_2021_v7_wgs84_bordermerged.csv'
muni_gdf = gpd.read_file('./data/municipios.shp')
muni_gdf['center_lon'] = muni_gdf.geometry.centroid.x
muni_gdf['center_lat'] = muni_gdf.geometry.centroid.y
res_gdf = read_process_csv_to_gdf(in_csv)
res_gdf['res_area_ha'] = res_gdf['area_ha']
joined_gdf = gpd.sjoin(res_gdf, muni_gdf, predicate='within', how='inner')
muni_res_stats = sjoin_summarize(res_gdf, muni_gdf, 'cd_mun')
muni_res_stats = muni_res_stats.join(muni_gdf.set_index('cd_mun')[['area_ha', 'center_lon','center_lat']])

In [ ]:
cattle = pd.read_csv('./data/cattle.csv', index_col=0)
crops = pd.read_csv('./data/crops.csv', header=[0,1], index_col=0)
pib = pd.read_csv('./data/pib.csv', header=[0,1], index_col=0)
pop = pd.read_csv('./data/pop.csv',index_col=0)
cattle.columns = (pd.MultiIndex.from_product([cattle.columns, ['cattle']]))
pop.columns = (pd.MultiIndex.from_product([pop.columns, ['pop']]))
biome_df = pd.read_csv('./data/biome.csv', index_col=0)
state_df = pd.read_csv('./data/mb_state.csv', index_col=0)

In [ ]:
irrigation = pd.read_csv('../irrigation/data/center_pivot_area_1985_2022.csv')
irrigation = irrigation.rename(columns={'mun_código':'cd_mun'})
irrigation = irrigation.iloc[:-9]
irrigation['cd_mun'] = irrigation['cd_mun'].astype(int)
irrigation = irrigation.set_index(
    'cd_mun'
    ).drop(
        columns=['mun_nome','UF_nome','UF_sigla','Região']
        )
irrigation.columns = 'irrigation_' + irrigation.columns
irrigation = irrigation.replace(',','', regex=True).astype(float)

In [ ]:
# Precip just needs a little processing
precip = pd.read_parquet('./data/pr_normal.parquet')
precip = (precip.set_index('code_muni')*30).reset_index()
precip['low_rain'] = precip['normal_mean']<100
precip = precip.rename(columns={'code_muni': 'cd_muni'})
precip_std = precip[['cd_muni', 'normal_mean']].groupby('cd_muni').std().rename(
    columns={'normal_mean':'normal_std'})
precip_min = precip.groupby('cd_muni').min()[['normal_mean']].rename(columns={'normal_mean': 'normal_min'})
precip_minmax = precip.groupby('cd_muni').max()['normal_mean'] - precip.groupby('cd_muni').min()['normal_mean']
precip_minmax.name = 'range'
precip = precip.groupby('cd_muni').mean().join(precip_std).join(precip_minmax).join(precip_min)
precip['std_div_mean'] = precip['normal_std']/precip['normal_mean']
precip.columns = 'precip_' + precip.columns
precip = precip.drop(columns='precip_month')

In [ ]:
def l1_processing(df):
    df = df.reset_index().set_index('cd_muni').pivot(columns='class_level_1').fillna(0)['2021'].rename(
        columns={
            '1. Forest': 'forest',
            '2. Non Forest Natural Formation': 'natural_nonforest',
            '3. Farming': 'farming',
            '4. Non vegetated area': 'non_veg',
            '5. Water and Marine Environment': 'water',
            '6. Not Observed': 'na',
        }
    )
    df['natural_total'] = df['forest'] + df['natural_nonforest']
    return df


In [ ]:
# Mapbiomas data
# Level 1
mb_df = pd.read_csv('./data/mb_lulc_cleaned.csv', index_col=[0,1])
mb_df = mb_df.fillna(0)
# Convert to percentages

mb_df = mb_df.div(mb_df.reset_index().groupby('cd_muni').sum()['1985'], axis=0) * 100

mb_diffs = mb_df[['2021']].copy()
mb_diffs['2021'] = mb_df['2021'] - mb_df['1985']
mb_diffs = l1_processing(mb_diffs)
mb_cur = l1_processing(mb_df['2021'])

# # Level 4
mb_l4_df = pd.read_csv('./data/mb_lulc_cleaned_level4.csv', index_col=[0,1])
mb_l4_df = mb_l4_df.fillna(0)
# Convert to percentages
mb_l4_df = mb_l4_df.div(mb_l4_df.reset_index().groupby('cd_muni').sum()['1985'], axis=0) * 100
mb_l4_cur = mb_l4_df['2021']
mb_l4_diffs = mb_l4_df['2021'] - mb_l4_df['1985']
mb_l4_diffs = mb_l4_diffs.reset_index().set_index('cd_muni').pivot(columns='class_level_4').fillna(0)[0]
mb_l4_cur = mb_l4_cur.reset_index().set_index('cd_muni').pivot(columns='class_level_4').fillna(0)['2021']

In [ ]:
mb_diffs['soy'] = mb_l4_diffs['3.2.1.1. Soybean']
mb_diffs['rice'] = mb_l4_diffs['3.2.1.3. Rice']
mb_diffs['pasture'] = mb_l4_diffs['3.1. Pasture']
mb_cur['soy'] = mb_l4_cur['3.2.1.1. Soybean']
mb_cur['rice'] = mb_l4_diffs['3.2.1.3. Rice']
mb_cur['pasture'] = mb_l4_cur['3.1. Pasture']

In [ ]:
mb_cur.columns = mb_cur.columns + '_current'
mb_diffs.columns = mb_diffs.columns + '_diff'

## Experiment 1: Random forest, 2021-data only

In [ ]:
preds_2021 = cattle.join(crops).join(pib).join(pop)['2021'].join(precip).join(mb_diffs).join(mb_cur).join(irrigation[['irrigation_2022']])

In [ ]:
full_df = preds_2021.join(muni_res_stats)

In [ ]:
full_df_density = full_df.div((full_df['area_ha']/100), axis=0)
full_df_density.columns = full_df_density.columns + '_density'
full_df_percapita = full_df.div((full_df['pop']), axis=0).drop(columns='gdp') # Duplicate
full_df_percapita.columns = full_df_percapita.columns + '_per_capita'

In [ ]:
full_df = full_df.join(full_df_density).join(full_df_percapita).join(biome_df).join(state_df)
# Remove a bad muni
full_df = full_df.loc[~full_df['biome'].isna()]

# Drop response vars
pred_df = full_df.copy()
for response_var in ['count','median','sum']:
    pred_df = pred_df.loc[:,~pred_df.columns.str.startswith(response_var)]

In [ ]:
full_df.to_csv('./data/full_df.csv')

### Exp 1a: All variables

In [ ]:
X = pred_df.fillna(0).drop(columns='biome')
feature_names = X.columns
# X = full_df[['cattle','count']].fillna(0)#full_df.drop(columns=['count','sum','median']).fillna(0)
y = full_df['count_density'].fillna(0)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=10)
reg = GradientBoostingRegressor(n_estimators=100)#svm.SVR()
reg.fit(X_train, y_train)
preds = reg.predict(X_test)
print(metrics.mean_absolute_error(y_test, preds))
print(metrics.root_mean_squared_error(y_test, preds))
print(metrics.r2_score(y_test, preds))

In [ ]:
plt.scatter(y_test, preds)
plt.xlabel('True')
plt.ylabel('Pred')

In [ ]:

from sklearn.inspection import permutation_importance
r = permutation_importance(reg, X_test, y_test,
                           n_repeats=30,
                           random_state=0)
for i in r.importances_mean.argsort()[::-1]:
    if r.importances_mean[i] - 2 * r.importances_std[i] > 0:
        print(f"{feature_names[i]:<8}"
              f"{r.importances_mean[i]:.3f}"
              f" +/- {r.importances_std[i]:.3f}")

In [ ]:
# Partial dependence
from sklearn.inspection import PartialDependenceDisplay
PartialDependenceDisplay.from_estimator(reg, X_train, ['cattle_density','pop_density', 'precip_normal_p90', 'forest'])

In [ ]:
# reg =GradientBoostingRegress,'administration_per_capita','agriculture_per_capita',
#               'services_per_capita','industry_per_capita']or(n_estimators=100)
# cross_val_score(reg, X_train, y_train, cv=5, scoring='r2')

### Exp 1b: Select vars

In [ ]:
var_select = [
    'cattle_density', 
    # 'gdp_per_capita',
    'pop_density',
    # 'forest',
    # 'farming',
    # 'corn_area_density',
    # 'soy_area_density',
    # 'administration_density',
    'precip_normal_mean',
    # 'precip_low_rain',
    'precip_normal_std',
    # 'precip_range',
    'irrigation_2022_density'
    ]

In [ ]:
X = pred_df[var_select].fillna(0)
feature_names = var_select
y = full_df['count_density'].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=11)
reg = GradientBoostingRegressor(n_estimators=100)#svm.SVR()
reg.fit(X_train, y_train)
preds = reg.predict(X_test)
print(metrics.mean_absolute_error(y_test, preds))
print(metrics.r2_score(y_test, preds))

In [ ]:
plt.scatter(y_test, preds)
plt.xlabel('True')
plt.ylabel('Pred')

In [ ]:
from sklearn.inspection import permutation_importance
r = permutation_importance(reg, X_train, y_train,
                           n_repeats=30,
                           random_state=0)
for i in r.importances_mean.argsort()[::-1]:
    if r.importances_mean[i] - 2 * r.importances_std[i] > 0:
        print(f"{feature_names[i]:<8}"
              f"{r.importances_mean[i]:.3f}"
              f" +/- {r.importances_std[i]:.3f}")

In [ ]:

from sklearn.inspection import PartialDependenceDisplay
for var in feature_names:
    PartialDependenceDisplay.from_estimator(reg, X_train, [var])

In [ ]:

PartialDependenceDisplay.from_estimator(reg, X_train, [['precip_normal_mean','precip_normal_std']])

## Exp 2: GAM

### Exp 2a: Select vars

In [ ]:
var_select = [
    'biome',
    'cattle_density', 
    # 'administration',
    # 'gdp',
    'pop_density',
    'precip_normal_mean',
    # 'precip_normal_std',
    'precip_std_div_mean',
    'irrigation_2022_density'
    # 'forest',
    # 'area_ha',
    ]

In [ ]:
X = pred_df[var_select].fillna(0)
X = X.loc[X['biome'] != 0]
X['biome'] = pd.Categorical(X['biome'])
biome_cats = dict( enumerate(X['biome'].cat.categories ) )
X['biome'] = X['biome'].cat.codes
y = full_df['count_density'].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=14)

In [ ]:
gam = pygam.GAM(pygam.f(0) + pygam.s(1) + pygam.s(2) + pygam.s(3) + pygam.s(4) + pygam.te(3,4), distribution='poisson', link='log')
# gam = pygam.PoissonGAM(pygam.f(0) + pygam.s(1) + pygam.s(2) + pygam.s(4) + pygam.te(3,4))
fitted_gam = gam.fit(X_train, y_train)
opt_gam = fitted_gam

In [ ]:
fitted_gam.summary()

In [ ]:
term_names = ['Biome', 'Cattle herd (density)', 'Population density', 'Mean precip','std. dev. precip', 'a','irrigation']
for i, term in enumerate(gam.terms):
    if term.isintercept or term.istensor:
        continue

    XX = opt_gam.generate_X_grid(term=i)
    pdep, confi = opt_gam.partial_dependence(term=i, X=XX, width=0.95)

    plt.figure()
    plt.plot(XX[:, term.feature], pdep)
    plt.plot(XX[:, term.feature], confi, c='r', ls='--')
    plt.title(term_names[i])
    plt.show()

In [ ]:
XX = opt_gam.generate_X_grid(term=5, meshgrid=True)
Z = opt_gam.partial_dependence(term=5, X=XX, meshgrid=True)

ax = plt.axes(projection='3d')
ax.plot_surface(XX[0], XX[1], Z, cmap='viridis')



In [ ]:

# Evaluate
gam_predict = opt_gam.predict(X_test)
plt.scatter(gam_predict, y_test)

In [ ]:
metrics.mean_absolute_error(gam_predict, y_test)

In [ ]:
metrics.r2_score(y_test, gam_predict)

In [ ]:
plt.scatter(gam_predict, opt_gam.deviance_residuals(X_test, y_test))

### Experiment 2b: Only cattle density

In [ ]:

var_select = [
    'biome',
    'cattle_density', 
    # 'pop_density',
    # 'precip_normal_mean',
    # 'precip_std_div_mean',
    # 'irrigation_2022_density'
    # 'forest',
    # 'area_ha',
    ]

In [ ]:
X = pred_df[var_select].fillna(0)
X = X.loc[X['biome'] != 0]
X['biome'] = pd.Categorical(X['biome'])
biome_cats = dict( enumerate(X['biome'].cat.categories ) )
X['biome'] = X['biome'].cat.codes
y = full_df['count_density'].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=14)

In [ ]:
gam = pygam.PoissonGAM(pygam.f(0) + pygam.s(1))
fitted_gam = gam.fit(X_train, y_train)
opt_gam = fitted_gam

In [ ]:
fitted_gam.summary()

In [ ]:

for i, term in enumerate(gam.terms):
    if term.isintercept:
        continue

    XX = opt_gam.generate_X_grid(term=i)
    pdep, confi = opt_gam.partial_dependence(term=i, X=XX, width=0.95)

    plt.figure()
    plt.plot(XX[:, term.feature], pdep)
    plt.plot(XX[:, term.feature], confi, c='r', ls='--')
    plt.title(repr(term))
    plt.show()

### Exp 2c: Modeling raw counts, not density

In [ ]:
var_select = [
    'biome',
    'cattle_density', 
    'administration',
    'agriculture',
    'gdp_per_capita',
    'pop_density',
    'precip_normal_mean',
    'precip_normal_std',
    'precip_std_div_mean',
    'forest',
    'corn_area_density',
    'soy_area_density',
    'area_ha'
    ]

In [ ]:
X = pred_df[var_select].loc[~pred_df['area_ha'].isna()].fillna(0)
X = X.loc[X['biome'] != 0]
X['biome'] = pd.Categorical(X['biome'])
biome_cats = dict( enumerate(X['biome'].cat.categories ) )
X['biome'] = X['biome'].cat.codes
y = full_df['count'].loc[~pred_df['area_ha'].isna()].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=14)

In [ ]:
gam = pygam.PoissonGAM(pygam.f(0, lam=1.0) + pygam.s(1, lam=1000) + pygam.s(2, lam=1000) + pygam.s(3, lam=1000) + pygam.s(4, lam=1000) + pygam.te(3,4, lam=100.0) + pygam.s(5, lam=1000) + pygam.s(6, lam=1000) + pygam.s(7, lam=1000))# + pygam.s(5))
fitted_gam = gam.fit(X_train, y_train, exposure=X_train['area_ha'])
opt_gam = fitted_gam
fitted_gam.summary()

In [ ]:
# # lams = np.random.rand(1000, len(gam.terms)) # random points on [0, 1], with shape (100, 3)
# # lams = lams * 2 - 1 # shift values to -2, 2
# # lams = 10 ** lams # transforms values to 1e-3, 1e3
# lam_options = np.array([0.6, 0.8])
# lams = [lam_options] * len(gam.terms)
# gam = pygam.PoissonGAM(pygam.f(0) + pygam.s(1) + pygam.s(2) + pygam.te(3,4))
# opt_gam = gam.gridsearch(X_train, y_train, exposure=X_train['area_ha'], lam=lams)
# # opt_gam = gam.gridsearch(X_train, y_train, exposure=X_train['area_ha'], lam=lams)
# print(opt_gam.summary())


In [ ]:
for i, term in enumerate(gam.terms):
    if term.isintercept:
        continue

    XX = opt_gam.generate_X_grid(term=i)
    pdep, confi = opt_gam.partial_dependence(term=i, X=XX, width=0.95)

    plt.figure()
    plt.plot(XX[:, term.feature], pdep)
    plt.plot(XX[:, term.feature], confi, c='r', ls='--')
    plt.title(repr(term))
    plt.show()

In [ ]:
XX = opt_gam.generate_X_grid(term=5, meshgrid=True)
Z = opt_gam.partial_dependence(term=5, X=XX, meshgrid=True)

ax = plt.axes(projection='3d')
ax.plot_surface(XX[0], XX[1], Z, cmap='viridis')

In [ ]:
# Evaluate
gam_predict = fitted_gam.predict(X_test, exposure=X_test['area_ha'])
plt.scatter(gam_predict, y_test)
plt.axline((0,0),slope=1)

plt.xlim(0, 4000)
plt.ylim(0, 4000)

In [ ]:
metrics.mean_absolute_error(gam_predict, y_test)

In [ ]:
np.mean(gam_predict)

In [ ]:
np.var(gam_predict)

In [ ]:
metrics.r2_score(y_test,gam_predict)

In [ ]:
plt.hist(opt_gam.deviance_residuals(X_test, y_test/X_test['area_ha']))

In [ ]:
plt.scatter(gam_predict, opt_gam.deviance_residuals(X_test, y_test/X_test['area_ha']))

## Experiment 3: Time series

### Exp 3a: Raw data

In [ ]:
decadal = cattle.join(crops).join(pib).join(pop)[['1980','1990','2000','2010','2020']]
decadal.columns = decadal.columns.swaplevel(0,1)
decadal.columns = decadal.columns.map('_'.join).str.strip('_')
decadal = decadal.fillna(0)

In [ ]:
full_decadal = decadal.join(muni_res_stats)

In [ ]:
full_decadal_density = full_decadal.div((full_decadal['area_ha']/100), axis=0)
full_decadal_density.columns = full_decadal_density.columns + '_density'
full_decadal_percapita = full_decadal.div((full_decadal['pop_2020']), axis=0)
full_decadal_percapita.columns = full_decadal_percapita.columns + '_per_capita'

In [ ]:
full_decadal = full_decadal.join(full_decadal_density).join(full_decadal_percapita)

# Drop response vars
preds_decadal = full_decadal.copy()
for response_var in ['count','median','sum']:
    preds_decadal = preds_decadal.loc[:,~preds_decadal.columns.str.startswith(response_var)]

In [ ]:
X = preds_decadal.fillna(0)
feature_names = X.columns
# X = full_df[['cattle','count']].fillna(0)#full_df.drop(columns=['count','sum','median']).fillna(0)
y = full_decadal['count_density'].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=10)
reg = GradientBoostingRegressor(n_estimators=100)#svm.SVR()
reg.fit(X_train, y_train)
preds = reg.predict(X_test)
print(metrics.mean_absolute_error(y_test, preds))
print(metrics.root_mean_squared_error(y_test, preds))
print(metrics.r2_score(y_test, preds))

In [ ]:
from sklearn.inspection import permutation_importance
r = permutation_importance(reg, X_test, y_test,
                           n_repeats=10,
                           random_state=0)
for i in r.importances_mean.argsort()[::-1]:
    if r.importances_mean[i] - 2 * r.importances_std[i] > 0:
        print(f"{feature_names[i]:<8}"
              f"{r.importances_mean[i]:.3f}"
              f" +/- {r.importances_std[i]:.3f}")

In [ ]:
plt.scatter(y_test, preds)
plt.xlabel('True')
plt.ylabel('Pred')

### Exp 2b: Selective difference variables

In [ ]:
cattle_diff = (preds_decadal['cattle_2000_density'])
cattle_cur = preds_decadal['cattle_2020_density'] 
pop_cur = preds_decadal['pop_2020_density']
soy_diff = preds_decadal['soy_area_2020_density'] - preds_decadal['soy_area_2000_density']
gdp_diff = preds_decadal['gdp_per_capita_2020'] - preds_decadal['gdp_per_capita_2010']
gdp_cur = preds_decadal['gdp_per_capita_2020']

In [ ]:
diff_pred_df = pd.concat([cattle_diff,
                          cattle_cur,
                          pop_cur,
                          soy_diff,
                          gdp_diff,
                          gdp_cur], axis=1)
diff_pred_df.columns = ['cattle_density_diff', 'cattle_density','pop_density','soy_density_diff', 'gdp_diff','gdp_cur']
feature_select = [
    'cattle_density_diff',
    'cattle_density',
    'pop_density',
    'soy_density_diff',
    'gdp_diff',
    'gdp_cur']

In [ ]:
X = diff_pred_df[feature_select].fillna(0)
feature_names = X.columns
# X = full_df[['cattle','count']].fillna(0)#full_df.drop(columns=['count','sum','median']).fillna(0)
y = full_decadal['count_density'].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=10)
reg = GradientBoostingRegressor(n_estimators=100)#svm.SVR()
reg.fit(X_train, y_train)
preds = reg.predict(X_test)
print(metrics.mean_absolute_error(y_test, preds))
print(metrics.root_mean_squared_error(y_test, preds))
print(metrics.r2_score(y_test, preds))

In [ ]:

from sklearn.inspection import permutation_importance
r = permutation_importance(reg, X_test, y_test,
                           n_repeats=30,
                           random_state=0)
for i in r.importances_mean.argsort()[::-1]:
    if r.importances_mean[i] - 2 * r.importances_std[i] > 0:
        print(f"{feature_names[i]:<8}"
              f"{r.importances_mean[i]:.3f}"
              f" +/- {r.importances_std[i]:.3f}")